# 05 - Model Registry and Deployment (SageMaker SDK v3)

Registers the model produced by notebook 04, deploys a real-time endpoint with data capture, invokes it, and runs batch transform.

This version uses explicit SageMaker control-plane APIs for model creation and registration plus the v3 `sagemaker.core.transformer.Transformer`. It avoids the unstable repack implementation in early `ModelBuilder` v3 releases and contains no SDK v2 imports.


In [1]:
import sys, json, time, os, shutil, tempfile, tarfile
from pathlib import Path

# Run this notebook from <repository>/notebooks.
REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import boto3
import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
import awswrangler as wr
from botocore.exceptions import ClientError

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris
from src import config, data_utils, features, modeling

session = Session()
role = get_execution_role()
REGION = session.boto_region_name
BUCKET = os.environ.get("CHURN_BUCKET", session.default_bucket())

sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)

FRAMEWORK_VERSION = "1.4-2-py312"
INSTANCE_TYPE = "ml.m5.large"

def stage_source():
    build = Path(tempfile.mkdtemp(prefix="churn-inference-"))
    shutil.copytree(REPO_ROOT / "src", build / "src", ignore=shutil.ignore_patterns("__pycache__"))
    for script in (REPO_ROOT / "scripts").glob("*.py"):
        shutil.copy2(script, build / script.name)
    # Match the environment that created model.joblib. Unpinned packages can
    # upgrade scikit-learn/NumPy inside the endpoint and make the pickle unloadable.
    (build / "requirements.txt").write_text(
        f"numpy=={np.__version__}\n"
        f"scipy=={scipy.__version__}\n"
        f"pandas=={pd.__version__}\n"
        f"scikit-learn=={sklearn.__version__}\n"
        f"joblib=={joblib.__version__}\n"
    )
    # The Python 3.12 sklearn container's auto-generated setup.py can omit a
    # standalone inference.py from the wheel. Declare it explicitly.
    (build / "setup.py").write_text(
        "from setuptools import find_packages, setup\n"
        "setup(name='churn-inference', version='1.0.0', "
        "py_modules=['inference'], packages=find_packages(), "
        f"install_requires=['numpy=={np.__version__}', 'scipy=={scipy.__version__}', "
        f"'pandas=={pd.__version__}', 'scikit-learn=={sklearn.__version__}', "
        f"'joblib=={joblib.__version__}'])\n"
    )
    # The training artifact produced by notebook 04 may pickle the BitGenerator
    # class rather than its name. Apply the same compatibility shim used there
    # before model_fn calls joblib.load during the endpoint health check.
    with (build / "inference.py").open("a") as inference_file:
        inference_file.write(
            "\n# NumPy model-artifact compatibility shim\n"
            "from numpy.random import MT19937, PCG64, Philox, SFC64\n"
            "import numpy.random._pickle as _numpy_pickle\n"
            "for _bit_generator in (MT19937, PCG64, Philox, SFC64):\n"
            "    _numpy_pickle.BitGenerators.setdefault(_bit_generator, _bit_generator)\n"
        )
    return str(build)

latest_key = f"{config.MODEL_PREFIX}/latest/latest.json"

def load_or_recover_manifest():
    try:
        return json.loads(s3.get_object(Bucket=BUCKET, Key=latest_key)["Body"].read())
    except s3.exceptions.NoSuchKey:
        print(f"s3://{BUCKET}/{latest_key} is missing; recovering the newest completed churn training job")

    jobs = sm.list_training_jobs(
        NameContains="churn-train", StatusEquals="Completed",
        SortBy="CreationTime", SortOrder="Descending", MaxResults=20,
    )["TrainingJobSummaries"]
    if not jobs:
        raise RuntimeError(
            "No completed 'churn-train' job was found. Run section 8 of notebook 04 first, "
            "then rerun this cell."
        )

    job = sm.describe_training_job(TrainingJobName=jobs[0]["TrainingJobName"])
    recovered = {
        "model_data": job["ModelArtifacts"]["S3ModelArtifacts"],
        "model_type": job.get("HyperParameters", {}).get("model-type", "unknown").strip('"'),
        # Used only by Model Monitor. The deployed model retains its own trained threshold.
        "threshold": 0.5,
        "training_job_name": job["TrainingJobName"],
        "recovered": True,
    }
    evaluation_key = f"{config.MODEL_PREFIX}/latest/evaluation.json"
    try:
        s3.head_object(Bucket=BUCKET, Key=evaluation_key)
        recovered["evaluation"] = f"s3://{BUCKET}/{evaluation_key}"
    except Exception as error:
        if getattr(error, "response", {}).get("Error", {}).get("Code") not in ("404", "NoSuchKey", "NotFound"):
            raise
        print("evaluation.json is also missing; registration will continue without model metrics")

    # Repair the manifest so notebooks 05 and 06 use the same recovered artifact.
    s3.put_object(Bucket=BUCKET, Key=latest_key, Body=json.dumps(recovered, indent=2).encode())
    return recovered

latest = load_or_recover_manifest()
print("SDK v3 session:", REGION, BUCKET)
latest

from sagemaker.core.transformer import Transformer


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
SDK v3 session: us-east-1 sagemaker-us-east-1-025824828736


## 1. Build and register the model

The code explicitly repacks `inference.py` and the project package into the trained artifact. This avoids the broken temporary-directory behavior in early SDK v3 `ModelBuilder` releases. The framework version intentionally matches notebook 04.


In [2]:
try:
    group = sm.describe_model_package_group(ModelPackageGroupName=config.MODEL_PACKAGE_GROUP)
    print("model package group already exists:", group["ModelPackageGroupArn"])
except ClientError as error:
    code = error.response.get("Error", {}).get("Code")
    message = error.response.get("Error", {}).get("Message", "")
    if code not in ("ValidationException", "ResourceNotFound") and "does not exist" not in message.lower():
        raise
    group = sm.create_model_package_group(
        ModelPackageGroupName=config.MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription="Customer churn classifier",
    )
    print("created model package group:", group["ModelPackageGroupArn"])

image_uri = image_uris.retrieve(
    framework="sklearn", region=REGION, version=FRAMEWORK_VERSION,
    py_version="py3", instance_type=INSTANCE_TYPE, image_scope="inference",
)
source_dir = stage_source()

def repack_model(model_data, source_directory):
    work = Path(tempfile.mkdtemp(prefix="churn-repack-"))
    original = work / "original-model.tar.gz"
    extracted = work / "model"
    extracted.mkdir()

    bucket, key = model_data[5:].split("/", 1)
    s3.download_file(bucket, key, str(original))
    with tarfile.open(original, "r:gz") as archive:
        archive.extractall(extracted, filter="data")

    code_dir = extracted / "code"
    code_dir.mkdir(exist_ok=True)
    for item in Path(source_directory).iterdir():
        destination = code_dir / item.name
        if item.is_dir():
            shutil.copytree(item, destination, dirs_exist_ok=True)
        else:
            shutil.copy2(item, destination)

    if not (code_dir / "inference.py").is_file():
        raise RuntimeError("Repacked artifact is missing code/inference.py")

    repacked = work / "repacked-model.tar.gz"
    with tarfile.open(repacked, "w:gz") as archive:
        for item in extracted.iterdir():
            archive.add(item, arcname=item.name)

    key = f"{config.MODEL_PREFIX}/repacked/{int(time.time())}/model.tar.gz"
    s3.upload_file(str(repacked), BUCKET, key)
    return f"s3://{BUCKET}/{key}"

repacked_model_data = repack_model(latest["model_data"], source_dir)
timestamp = time.strftime("%Y%m%d-%H%M%S")
model_name = f"churn-model-{timestamp}"
container = {
    "Image": image_uri,
    "ModelDataUrl": repacked_model_data,
    "Environment": {
        "SAGEMAKER_PROGRAM": "inference.py",
        # The model archive is extracted under /opt/ml/model. Point script mode
        # at its code subdirectory; do not use the S3 model URI here.
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
        "SAGEMAKER_REGION": REGION,
    },
}

sm.create_model(ModelName=model_name, PrimaryContainer=container, ExecutionRoleArn=role)

package_request = {
    "ModelPackageGroupName": config.MODEL_PACKAGE_GROUP,
    "ModelPackageDescription": f"{latest['model_type']} churn model, threshold {latest['threshold']}",
    "ModelApprovalStatus": "Approved",
    "InferenceSpecification": {
        "Containers": [container],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
        "SupportedRealtimeInferenceInstanceTypes": [INSTANCE_TYPE],
        "SupportedTransformInstanceTypes": [INSTANCE_TYPE],
    },
}
if latest.get("evaluation"):
    package_request["ModelMetrics"] = {
        "ModelQuality": {
            "Statistics": {"ContentType": "application/json", "S3Uri": latest["evaluation"]}
        }
    }
package = sm.create_model_package(**package_request)
print("registered:", package["ModelPackageArn"])
print("SageMaker model:", model_name)
print("repacked artifact:", repacked_model_data)


model package group already exists: arn:aws:sagemaker:us-east-1:025824828736:model-package-group/churn-model-group
registered: arn:aws:sagemaker:us-east-1:025824828736:model-package/churn-model-group/6
SageMaker model: churn-model-20261019-121409
repacked artifact: s3://sagemaker-us-east-1-025824828736/churn/models/repacked/1792412049/model.tar.gz


## 2. Deploy or update the real-time endpoint

The endpoint captures every request and response for notebook 06. Re-running this cell updates an existing endpoint.


In [3]:
endpoint_config_name = f"{config.ENDPOINT_NAME}-{time.strftime('%Y%m%d-%H%M%S')}"
sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": INSTANCE_TYPE,
        "InitialVariantWeight": 1.0,
    }],
    DataCaptureConfig={
        "EnableCapture": True,
        "InitialSamplingPercentage": 100,
        "DestinationS3Uri": f"s3://{BUCKET}/{config.MONITOR_PREFIX}/data-capture",
        "CaptureOptions": [{"CaptureMode": "Input"}, {"CaptureMode": "Output"}],
        "CaptureContentTypeHeader": {"CsvContentTypes": ["text/csv"]},
    },
)

try:
    endpoint_description = sm.describe_endpoint(EndpointName=config.ENDPOINT_NAME)
except ClientError as error:
    if error.response.get("Error", {}).get("Code") != "ValidationException":
        raise
    endpoint_description = None

if endpoint_description is None:
    sm.create_endpoint(EndpointName=config.ENDPOINT_NAME, EndpointConfigName=endpoint_config_name)
    print("creating endpoint")
else:
    status = endpoint_description["EndpointStatus"]
    print("existing endpoint status:", status)
    if status == "InService":
        sm.update_endpoint(EndpointName=config.ENDPOINT_NAME, EndpointConfigName=endpoint_config_name)
        print("updating endpoint")
    elif status in ("Creating", "Updating", "SystemUpdating", "RollingBack"):
        print("endpoint operation already in progress; waiting without submitting another update")
    elif status in ("Failed", "OutOfService", "UpdateRollbackFailed"):
        reason = endpoint_description.get("FailureReason", "No failure reason returned")
        raise RuntimeError(
            f"Endpoint is {status}: {reason}\n"
            "AWS cannot update a failed endpoint. Preserve this reason, then delete the endpoint "
            "and rerun this cell only after correcting the underlying container/configuration error."
        )
    else:
        raise RuntimeError(f"Endpoint is in unsupported state {status}: {endpoint_description}")

sm.get_waiter("endpoint_in_service").wait(
    EndpointName=config.ENDPOINT_NAME,
    WaiterConfig={"Delay": 30, "MaxAttempts": 120},
)
print("endpoint ready:", config.ENDPOINT_NAME)


creating endpoint
endpoint ready: churn-endpoint


## 3. Invoke the endpoint


In [4]:
production = data_utils.load_processed("production")

def invoke(rows: pd.DataFrame, inference_ids=None):
    out = []
    lines = rows[features.MODEL_FEATURES].to_csv(header=False, index=False).strip().split("\n")
    for i, line in enumerate(lines):
        kwargs = {"InferenceId": str(inference_ids[i])} if inference_ids is not None else {}
        response = runtime.invoke_endpoint(
            EndpointName=config.ENDPOINT_NAME, ContentType="text/csv", Accept="text/csv",
            Body=line.encode(), **kwargs,
        )
        out.append(response["Body"].read().decode().strip().split(","))
    return pd.DataFrame(out, columns=["probability", "prediction", "risk_tier"]).astype(
        {"probability": float, "prediction": int}
    )

sample = production.head(5)
pd.concat([sample[["customer_id", "churn"]].reset_index(drop=True), invoke(sample)], axis=1)


[10/19/26 12:23:14] INFO     Initializing a Ray instance                                               ]8;id=7440671;file:///opt/conda/lib/python3.12/site-packages/awswrangler/distributed/ray/_core.py\_core.py]8;;\:]8;id=7440672;file:///opt/conda/lib/python3.12/site-packages/awswrangler/distributed/ray/_core.py#202\202]8;;\

2026-10-19 12:23:16,418	WARNING services.py:2213 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 906964992 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=2.16gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-10-19 12:23:18,598	INFO worker.py:2012 -- Started a local Ray instance.
/opt/conda/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


,customer_id,churn,probability,prediction,risk_tier
0,1,1,1.000000,1,high
1,2,0,1.000000,1,high
2,3,0,0.999981,1,high
3,4,0,0.999999,1,high
4,5,0,0.999999,1,high


## 4. Batch transform


In [5]:
input_path = f"s3://{BUCKET}/churn/batch-input/production_features.csv"
output_path = f"s3://{BUCKET}/churn/batch-output/raw"
wr.s3.to_csv(production[features.MODEL_FEATURES], path=input_path, header=False, index=False)

transformer = Transformer(
    model_name=model_name,
    instance_count=1,
    instance_type=INSTANCE_TYPE,
    output_path=output_path,
    accept="text/csv",
    assemble_with="Line",
    strategy="MultiRecord",
    max_payload=6,
    sagemaker_session=session,
)
transformer.transform(data=input_path, content_type="text/csv", split_type="Line", wait=True, logs=True)


[10/19/26 12:23:27] INFO     Creating transform job with name:                                   ]8;id=7440679;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/transformer.py\transformer.py]8;;\:]8;id=7440680;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/transformer.py#403\403]8;;\
                             churn-model-20261019-121409-2026-10-19-12-23-27-296                                   

Output()

[10/19/26 12:26:59] INFO     churn-model-20261019-121409-2026-10-19-12-23-27-296/i-043549a62dd8e ]8;id=7440687;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7440688;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             a94a-1792412746:                                                                      
                             2026-10-19 12:26:51,445 INFO - sagemaker-containers - No GPUs                         
                             detected (normal if no gpus installed)                                                

                    INFO     churn-model-20261019-121409-2026-10-19-12-23-27-296/i-043549a62dd8e ]8;id=7440693;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7440694;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             a94a-1792412746:                                                                      
                             2026-10-19 12:26:51,447 INFO - sagemaker-containers - No GPUs                         
                             detected (normal if no gpus installed)                                                

[10/19/26 12:27:00] INFO     churn-model-20261019-121409-2026-10-19-12-23-27-296/i-043549a62dd8e ]8;id=7440699;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7440700;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             a94a-1792412746:                                                                      
                             2026-10-19 12:26:51,448 INFO - sagemaker-containers - nginx config:                   

...

                    INFO     churn-model-20261019-121409-2026-10-19-12-23-27-296/i-043549a62dd8e ]8;id=7442871;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7442872;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             a94a-1792412746:                                                                      
                             Exception ignored in: <function _after_fork at 0x7f34838a4c20>                        

                    INFO     churn-model-20261019-121409-2026-10-19-12-23-27-296/i-043549a62dd8e ]8;id=7442877;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7442878;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             a94a-1792412746:                                                                      
                             Traceback (most recent call last):                                                    
                               File "/usr/lib/python3.12/threading.py", line 1691, in                              
                             _after_fork                                                                           
                                 thread._set_tstate_lock()                                                         
                               File "/usr/lib/python3.12/threading.py", line 1051, in                              
                             _set_tstate_lock                                                                      
                                 if not self.daemon:                                                               
                                        ^^^^^^^^^^^                                                                
                               File "/usr/lib/python3.12/threading.py", line 1246, in daemon                       
                                 assert self._initialized, "Thread.__init__() not called"                          
                                        ^^^^^^^^^^^^^^^^^                                                          

                    INFO     churn-model-20261019-121409-2026-10-19-12-23-27-296/i-043549a62dd8e ]8;id=7442883;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7442884;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             a94a-1792412746:                                                                      
                             AssertionError: Thread.__init__() not called                                          

                    INFO     churn-model-20261019-121409-2026-10-19-12-23-27-296/i-043549a62dd8e ]8;id=7442889;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7442890;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32476\32476]8;;\
                             a94a-1792412746:                                                                      
                             169.254.255.130 - - [19/Oct/2026:12:27:21 +0000] "POST /invocations                   
                             HTTP/1.1" 200 1028995 "-" "Go-http-client/1.1"                                        

                    INFO     Final Resource Status: Completed                                    ]8;id=7442896;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=7442897;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#32482\32482]8;;\

In [6]:
out = wr.s3.read_csv(
    f"{output_path}/production_features.csv.out", header=None,
    names=["probability", "prediction", "risk_tier"],
)
scored = pd.concat([production[["customer_id", "churn"]].reset_index(drop=True), out], axis=1)
wr.s3.to_csv(scored, path=f"s3://{BUCKET}/churn/batch-output/scored_customers.csv", index=False)

print(scored["risk_tier"].value_counts())
print({k: round(v, 4) for k, v in modeling.metrics(
    scored["churn"], scored["probability"], latest["threshold"]
).items() if k in ("roc_auc", "precision", "recall", "top_decile_precision")})
scored.sort_values("probability", ascending=False).head(10)


risk_tier
high      62879
low        1326
medium      169
Name: count, dtype: int64
{'roc_auc': 0.6916, 'precision': 0.4834, 'recall': 0.9993, 'top_decile_precision': 0.6256}


,customer_id,churn,probability,prediction,risk_tier
64373,64374,1,1.0,1,high
0,1,1,1.0,1,high
1,2,0,1.0,1,high
64372,64373,1,1.0,1,high
64371,64372,1,1.0,1,high
64369,64370,1,1.0,1,high
5,6,0,1.0,1,high
6,7,1,1.0,1,high
7,8,0,1.0,1,high
8,9,0,1.0,1,high


## 5. Registry contents and cleanup


In [7]:
pkgs = sm.list_model_packages(
    ModelPackageGroupName=config.MODEL_PACKAGE_GROUP,
    SortBy="CreationTime", SortOrder="Descending",
)
pd.DataFrame(pkgs["ModelPackageSummaryList"])[
    ["ModelPackageVersion", "ModelApprovalStatus", "CreationTime", "ModelPackageArn"]
]


,ModelPackageVersion,ModelApprovalStatus,CreationTime,ModelPackageArn
0,6,Approved,2026-10-19 12:14:10.811000+00:00,arn:aws:sagemaker:us-east-1:025824828736:model...
1,5,Approved,2026-10-19 11:36:20.070000+00:00,arn:aws:sagemaker:us-east-1:025824828736:model...
2,4,Approved,2026-10-19 11:07:15.903000+00:00,arn:aws:sagemaker:us-east-1:025824828736:model...
3,3,Approved,2026-10-19 10:30:09.539000+00:00,arn:aws:sagemaker:us-east-1:025824828736:model...
4,2,Approved,2026-10-19 10:21:49.438000+00:00,arn:aws:sagemaker:us-east-1:025824828736:model...
5,1,Approved,2026-10-19 09:43:48.116000+00:00,arn:aws:sagemaker:us-east-1:025824828736:model...


To stop endpoint charges after the demo:

```python
sm.delete_endpoint(EndpointName=config.ENDPOINT_NAME)
```

Do not run that cleanup before notebook 06.


In [8]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>


I0000 00:00:1792412920.020929   20210 chttp2_transport.cc:1182] ipv4:169.255.255.2:46105: Got goaway [2] err=UNAVAILABLE:GOAWAY received; Error code: 2; Debug Text: Cancelling all calls {created_time:"2026-10-19T12:28:40.020925552+00:00", http2_error:2, grpc_status:14}
*** SIGTERM received at time=1792412922 on cpu 1 ***
PC: @     0x7f363965f2d2  (unknown)  epoll_wait
[2026-10-19 12:28:42,499 E 19598 19598] logging.cc:474: *** SIGTERM received at time=1792412922 on cpu 1 ***
[2026-10-19 12:28:42,499 E 19598 19598] logging.cc:474: PC: @     0x7f363965f2d2  (unknown)  epoll_wait
